In [1]:
from pathlib import Path
print(Path.cwd())

/Users/lohit109


In [1]:
import sys
from pathlib import Path

print("Python:", sys.executable)
print("Folder:", Path.cwd())

Python: /Users/lohit109/anaconda3/bin/python
Folder: /Users/lohit109/ticket-classifier


In [2]:
import pandas as pd
import sklearn
import matplotlib.pyplot as plt

print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Ready to build the ticket classifier!")

Pandas: 1.5.3
Scikit-learn: 1.3.0
Ready to build the ticket classifier!


In [3]:
import pandas as pd

base_url = (
    "https://raw.githubusercontent.com/"
    "PolyAI-LDN/task-specific-datasets/master/banking_data"
)

train_df = pd.read_csv(f"{base_url}/train.csv")

print("Training examples:", len(train_df))
print("Columns:", train_df.columns.tolist())

train_df.head()

Training examples: 10003
Columns: ['text', 'category']


,text,category
0,I am still waiting on my card?,card_arrival
1,What can I do if my card still hasn't arrived ...,card_arrival
2,I have been waiting over a week. Is the card s...,card_arrival
3,Can I track my card while it is in the process...,card_arrival
4,"How do I know if I will get my card, or if it ...",card_arrival


In [4]:
print("Number of categories:", train_df["category"].nunique())

print("\nMissing values:")
print(train_df.isna().sum())

print("\nRepeated messages:")
print(train_df.duplicated(subset=["text"]).sum())

category_counts = train_df["category"].value_counts()

print("\nExamples per category:")
print(category_counts.describe())

print("\n10 most common categories:")
print(category_counts.head(10))

Number of categories: 77

Missing values:
text        0
category    0
dtype: int64

Repeated messages:
0

Examples per category:
count     77.000000
mean     129.909091
std       32.942207
min       35.000000
25%      112.000000
50%      127.000000
75%      159.000000
max      187.000000
Name: category, dtype: float64

10 most common categories:
card_payment_fee_charged                            187
direct_debit_payment_not_recognised                 182
balance_not_updated_after_cheque_or_cash_deposit    181
wrong_amount_of_cash_received                       180
cash_withdrawal_charge                              177
transaction_charged_twice                           175
declined_cash_withdrawal                            173
transfer_fee_charged                                172
transfer_not_received_by_recipient                  171
balance_not_updated_after_bank_transfer             171
Name: category, dtype: int64


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    train_df["text"],
    train_df["category"],
    test_size=0.20,
    random_state=42,
    stratify=train_df["category"]
)

print("Training examples:", len(X_train))
print("Validation examples:", len(X_val))
print("Training categories:", y_train.nunique())
print("Validation categories:", y_val.nunique())

Training examples: 8002
Validation examples: 2001
Training categories: 77
Validation categories: 77


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)

print("Training matrix:", X_train_tfidf.shape)
print("Validation matrix:", X_val_tfidf.shape)

Training matrix: (8002, 8903)
Validation matrix: (2001, 8903)


In [7]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model.fit(X_train_tfidf, y_train)

print("Training complete!")
print("Categories learned:", len(model.classes_))

Training complete!
Categories learned: 77


In [8]:
from sklearn.metrics import accuracy_score, f1_score

y_pred = model.predict(X_val_tfidf)

accuracy = accuracy_score(y_val, y_pred)
macro_f1 = f1_score(y_val, y_pred, average="macro")

print(f"Validation accuracy: {accuracy:.2%}")
print(f"Validation macro-F1: {macro_f1:.4f}")

Validation accuracy: 84.36%
Validation macro-F1: 0.8409


In [9]:
results = pd.DataFrame({
    "message": X_val.to_numpy(),
    "actual": y_val.to_numpy(),
    "predicted": y_pred
})

errors = results[results["actual"] != results["predicted"]]

print("Correct predictions:", len(results) - len(errors))
print("Incorrect predictions:", len(errors))

with pd.option_context("display.max_colwidth", None):
    display(errors.head(10))

Correct predictions: 1688
Incorrect predictions: 313


,message,actual,predicted
1,What are the fees imposed when I make a transaction?,card_payment_fee_charged,cancel_transfer
7,My withdrawals were going smooth until I got declined today. What is going on?,declined_cash_withdrawal,declined_transfer
9,What do I need to do for a refund?,request_refund,Refund_not_showing_up
13,How can I reset my darned PIN number?,pin_blocked,change_pin
24,Why is there an uknown charge showing on my account?,card_payment_not_recognised,extra_charge_on_statement
30,What's up with the extra fee I got?,cash_withdrawal_charge,card_payment_fee_charged
37,Why has my payment not transferred?,failed_transfer,reverted_card_payment?
42,I need help verifying my identity.,unable_to_verify_identity,why_verify_identity
56,What currencies can be used for top up?,supported_cards_and_currencies,fiat_currency_support
66,what is this extra charge with my purchase?,card_payment_fee_charged,extra_charge_on_statement


In [10]:
confusion_pairs = (
    errors
    .groupby(["actual", "predicted"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

display(confusion_pairs.head(10))

,actual,predicted,count
113,get_disposable_virtual_card,disposable_card_limits,6
95,direct_debit_payment_not_recognised,card_payment_not_recognised,5
227,why_verify_identity,unable_to_verify_identity,4
174,top_up_by_bank_transfer_charge,top_up_by_card_charge,4
199,transfer_fee_charged,card_payment_fee_charged,4
180,top_up_failed,pending_top_up,4
191,topping_up_by_card,top_up_by_cash_or_cheque,3
102,exchange_via_app,exchange_charge,3
101,exchange_rate,card_payment_wrong_exchange_rate,3
140,pending_cash_withdrawal,declined_cash_withdrawal,3


In [11]:
model_c5 = LogisticRegression(
    C=5,
    max_iter=1000,
    random_state=42
)

model_c5.fit(X_train_tfidf, y_train)

y_pred_c5 = model_c5.predict(X_val_tfidf)

accuracy_c5 = accuracy_score(y_val, y_pred_c5)
macro_f1_c5 = f1_score(y_val, y_pred_c5, average="macro")

comparison = pd.DataFrame({
    "model": ["Logistic regression C=1", "Logistic regression C=5"],
    "validation_accuracy": [accuracy, accuracy_c5],
    "validation_macro_f1": [macro_f1, macro_f1_c5]
})

display(comparison.round(4))

,model,validation_accuracy,validation_macro_f1
0,Logistic regression C=1,0.8436,0.8409
1,Logistic regression C=5,0.8731,0.8745


In [12]:
from sklearn.pipeline import Pipeline

final_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2
    )),
    ("classifier", LogisticRegression(
        C=5,
        max_iter=1000,
        random_state=42
    ))
])

final_model.fit(
    train_df["text"],
    train_df["category"]
)

print("Final model trained on", len(train_df), "examples")

Final model trained on 10003 examples


In [13]:
test_df = pd.read_csv(f"{base_url}/test.csv")

test_predictions = final_model.predict(test_df["text"])

test_accuracy = accuracy_score(
    test_df["category"],
    test_predictions
)

test_macro_f1 = f1_score(
    test_df["category"],
    test_predictions,
    average="macro"
)

print("Test examples:", len(test_df))
print(f"Test accuracy: {test_accuracy:.2%}")
print(f"Test macro-F1: {test_macro_f1:.4f}")

Test examples: 3080
Test accuracy: 88.90%
Test macro-F1: 0.8894


In [14]:
from pathlib import Path
import joblib

model_dir = Path("models")
model_dir.mkdir(exist_ok=True)

model_path = model_dir / "ticket_classifier.joblib"
joblib.dump(final_model, model_path)

print("Model saved to:", model_path.resolve())

Model saved to: /Users/lohit109/ticket-classifier/models/ticket_classifier.joblib


In [15]:
loaded_model = joblib.load(model_path)

message = "My card hasn't arrived yet."
prediction = loaded_model.predict([message])

print("Message:", message)
print("Predicted category:", prediction[0])

Message: My card hasn't arrived yet.
Predicted category: card_arrival
